# Code Climber — Remote Kernel

This notebook turns your free Google Colab into a Python kernel that **Code Climber** can call into for the holds that need RDKit, PyTorch, or other libraries that don't run in the browser.

## How to use
1. Make sure a runtime is selected (top right). If you want GPU for the GNN holds: **Runtime → Change runtime type → T4 GPU**.
2. **Runtime → Run all** (Ctrl/Cmd-F9). First run takes 1–3 minutes while packages install.
3. The last cell prints a **URL** and a **token**. Paste both into the game's *Connect Colab* dialog.
4. Keep this tab open while you climb. If Colab disconnects (idle > ~90 min), re-run all cells and re-paste the new URL.


In [ ]:
# 1) Install dependencies. Heavy ones (torch, transformers) take a minute the first time.
!pip install -q fastapi uvicorn nest_asyncio rdkit xgboost
!pip install -q torch torch-geometric transformers

# 2) Grab cloudflared so we can expose the kernel via a public URL.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared
print('\n✓ deps installed')

In [ ]:
# 3) Define and start the runner. Same harness Code Climber uses locally,
#    just running on this Colab kernel where rdkit/torch/transformers are real.
import io, sys, secrets, threading, nest_asyncio
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

nest_asyncio.apply()

# Random per-session token. Keeps strangers from hitting your kernel even if your URL leaks.
TOKEN = secrets.token_urlsafe(16)

app = FastAPI()
app.add_middleware(
    CORSMiddleware, allow_origins=['*'],
    allow_methods=['*'], allow_headers=['*'],
)

@app.get('/ping')
def ping():
    return {'ok': True}

@app.post('/run')
async def run(req: Request):
    payload = await req.json()
    if payload.get('token') != TOKEN:
        return {'ok': False, 'err': 'unauthorized · token mismatch', 'stdout': ''}
    code = payload.get('code', '')
    verify = payload.get('verify', '')
    buf = io.StringIO()
    old, sys.stdout = sys.stdout, buf
    g = {'__name__': '__user__'}
    ok, err, phase = False, '', 'user'
    try:
        exec(compile(code, '<user>', 'exec'), g)
        phase = 'verify'
        vg = dict(g)
        vg['_stdout'] = buf.getvalue()
        vg['_user_globals'] = g
        exec(compile(verify, '<verify>', 'exec'), vg)
        ok = True
    except AssertionError as e:
        err = f'check failed · {e or "output did not match"}'
    except Exception as e:
        err = f'{phase} error · {type(e).__name__}: {e}'
    finally:
        sys.stdout = old
    return {'ok': ok, 'err': err, 'stdout': buf.getvalue()}

def _serve():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=_serve, daemon=True).start()
print('✓ runner listening on :8000')

In [ ]:
# 4) Open a public tunnel and print the URL + token to paste into Code Climber.
import subprocess, time, re, os

subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(1)

log_path = '/tmp/cloudflared.log'
if os.path.exists(log_path):
    os.remove(log_path)

p = subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--logfile', log_path],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

url = None
for _ in range(60):
    time.sleep(0.5)
    if os.path.exists(log_path):
        with open(log_path) as f:
            text = f.read()
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
        if m:
            url = m.group(0)
            break

print()
if url:
    print('=' * 64)
    print(' Paste these into Code Climber → Connect Colab')
    print('=' * 64)
    print()
    print(f'   URL:   {url}')
    print(f'   Token: {TOKEN}')
    print()
    print('=' * 64)
    print(' Keep this tab open. The URL dies if Colab idle-disconnects.')
    print('=' * 64)
else:
    print('✗ tunnel did not come up. Re-run this cell. If it keeps failing,')
    print('  the cloudflared binary download may have been blocked.')